# Train the from-scratch model on a Colab GPU

Runs Task 7 of the M2 plan (`docs/plans/2026-10-02-m2-scratch-model.md`) on a free GPU
instead of the M1 Mac: both training runs, `scratch-v1` (unweighted) and
`scratch-v1-inverse` (inverse-frequency class weights), with the same commands as
`scripts/train_scratch.py` documents. The notebook changes no training logic; it only
provides the machine.

**Before running:** Runtime > Change runtime type > T4 GPU. Then run the cells in order.

**Inputs:** the git branch (cell 2) and `data/clean/train.jsonl`, zipped as
`clean-train.zip` (cell 3; make it locally with
`cd data/clean && zip -j ../clean-train.zip train.jsonl`). Upload the zip to
`My Drive/newline-fixer/` or upload it when cell 3 asks.

**Outputs:** `experiments/runs/<run-id>/` (weights, git-ignored) and
`experiments/training/<run-id>.json` (the committed record). Both are copied to
`My Drive/newline-fixer/` after every epoch, so a disconnected session loses at most one
epoch, and zipped for download at the end. Back on the Mac, unzip into the repo root and
continue with step 3 of the Task 7 brief.

In [ ]:
import subprocess, sys
assert sys.version_info >= (3, 12), f"the project needs Python 3.12+, Colab has {sys.version}"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout or "NO GPU: set the runtime type to T4 GPU")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
REPO = "https://github.com/jalalhussein1982/newline-fixer.git"
BRANCH = "m2-scratch-model"  # the branch that holds the trainer

import os, shutil
if os.path.isdir("/content/newline-fixer"):
    shutil.rmtree("/content/newline-fixer")
!git clone --quiet --branch {BRANCH} {REPO} /content/newline-fixer
%cd /content/newline-fixer
# Colab's own torch carries CUDA; the project's uv.lock pins a CPU-only torch on Linux for CI,
# so install the package without resolving its extras.
!pip install --quiet --no-deps -e .
!git log --oneline -1

In [ ]:
import shutil
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/newline-fixer")
DRIVE.mkdir(parents=True, exist_ok=True)

zip_path = DRIVE / "clean-train.zip"
if not zip_path.exists():
    print("clean-train.zip not found in Drive; choose it from your computer")
    uploaded = files.upload()
    name = next(iter(uploaded))
    shutil.move(name, zip_path)  # Drive is another filesystem: rename would fail

Path("data/clean").mkdir(parents=True, exist_ok=True)
!unzip -q -o "{zip_path}" -d data/clean
!ls -la data/clean data/sets

In [ ]:
import json, shutil, subprocess, threading, time
from pathlib import Path

RUNS, RECORDS = Path("experiments/runs"), Path("experiments/training")


def sync(run_id: str) -> None:
    """Copy the run directory and the record to Drive (idempotent)."""
    if (RUNS / run_id).exists():
        shutil.copytree(RUNS / run_id, DRIVE / "runs" / run_id, dirs_exist_ok=True)
    if (RECORDS / f"{run_id}.json").exists():
        (DRIVE / "training").mkdir(exist_ok=True)
        shutil.copy(RECORDS / f"{run_id}.json", DRIVE / "training" / f"{run_id}.json")


def train(run_id: str, class_weights: str, every: int = 120) -> None:
    """scripts/train_scratch.py on CUDA, with a Drive sync every `every` seconds and at the end."""
    cmd = [
        "python", "-u", "scripts/train_scratch.py", "--run-id", run_id, "--seed", "1",  # -u: print each epoch as it ends
        "--class-weights", class_weights, "--device", "cuda",
    ]
    print(" ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    stop = threading.Event()

    def syncer() -> None:
        while not stop.wait(every):
            sync(run_id)

    threading.Thread(target=syncer, daemon=True).start()
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    stop.set()
    sync(run_id)
    if code != 0:
        raise RuntimeError(f"{run_id} exited with {code}")
    record = json.loads((RUNS / run_id / "run.json").read_text())
    print(f"\n{run_id}: best epoch {record['best_epoch']}, {record['seconds'] / 60:.1f} min, best {record['best']}")

In [ ]:
train("scratch-v1", "none")

In [ ]:
train("scratch-v1-inverse", "inverse")

In [ ]:
# Bundle both runs and their records; unzip at the repo root on the Mac.
!rm -f m2-runs.zip && zip -q -r m2-runs.zip experiments/runs/scratch-v1 experiments/runs/scratch-v1-inverse experiments/training/scratch-v1.json experiments/training/scratch-v1-inverse.json
shutil.copy("m2-runs.zip", DRIVE / "m2-runs.zip")
!ls -la m2-runs.zip
files.download("m2-runs.zip")

Back on the Mac, from the repo root:

```bash
unzip -o ~/Downloads/m2-runs.zip
```

Then step 3 of the Task 7 brief: choose the run by `best.V1_macro_f1` subject to
`best.V3_damage <= 0.0026`, point `experiments/runs/current` at it, and evaluate.